In [ ]:
# =====================================================================
# SIRA LABS - CHALLENGE QUOTIDIEN : ATTENTION SUR MESURE & SPAM SMS
# NOTEBOOK COMPLET - DÉCEMBRE 2025 / JUIN 2026
# =====================================================================

# ---------------------------------------------------------------------
# PART 1: SETUP & DATA LOADING
# ---------------------------------------------------------------------
print("--- Part 1: Installation des dépendances et chargement des données ---")
!pip install --quiet datasets evaluate transformers[sentencepiece] torch

import pandas as pd
from datasets import Dataset, load_dataset

# Chargement du jeu de données UCI SMS Spam depuis Hugging Face Hub
df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")

# Conversion en Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Séparation des données : 4000 échantillons pour l'entraînement, 1000 pour la validation
train_ds = hf_dataset.select(range(0, 4000))
val_ds   = hf_dataset.select(range(4000, 5000))

print(f"Données d'entraînement : {len(train_ds)} | Données de validation : {len(val_ds)}")
print(df.head())

# ---------------------------------------------------------------------
# PART 2: TOKENIZATION SETUP
# ---------------------------------------------------------------------
print("\n--- Part 2: Configuration de la Tokenisation ---")
from transformers import GPT2Tokenizer

model_name = "gpt2"
tokenizer  = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 n'a pas de token de padding par défaut, on lui assigne le token EOS
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],          # Colonne de texte source
        padding="max_length",     # Stratégie de padding uniforme
        truncation=True,          # Troncature activée
        max_length=64             # Longueur maximale fixée à 64
    )

# Application de la tokenisation par lots
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)

# ---------------------------------------------------------------------
# PART 3: PRE-TRAINED MODEL SETUP
# ---------------------------------------------------------------------
print("\n--- Part 3: Configuration du modèle pré-entraîné GPT-2 ---")
import torch
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,                          # Classification binaire (Spam vs Ham)
    pad_token_id=tokenizer.eos_token_id    # Assignation de l'ID de padding
)

# ---------------------------------------------------------------------
# PART 4: CUSTOM ATTENTION IMPLEMENTATION
# ---------------------------------------------------------------------
print("\n--- Part 4: Implémentation du mécanisme d'Attention sur Mesure ---")
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Facteur d'échelle : 1 / sqrt(embed_dim)
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        # Multiplication matricielle Q x K^T avec mise à l'échelle
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Softmax sur la dernière dimension pour obtenir les poids d'attention
        attn = F.softmax(scores, dim=-1)

        # Application des poids d'attention sur les Valeurs (V)
        return torch.matmul(attn, value), attn

class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)
        # Self-attention : Query = Key = Value = embed
        attn_output, _ = self.attn(embed, embed, embed)
        # Pooling : Calcul de la moyenne sur la dimension de la séquence (dim=1)
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)

# Préparation des données pour le modèle personnalisé
def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example["sms"], 
        max_length=64, 
        truncation=True, 
        padding="max_length"
    )
    return {"input_ids": tokens, "label": example["label"]}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

class SMSDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item["input_ids"], dtype=torch.long),
            'label': torch.tensor(item["label"], dtype=torch.long)
        }

# Création des DataLoaders
train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

# Paramètres et entraînement
vocab_size = tokenizer.vocab_size
embed_dim = 64
num_classes = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
model.to(device) # Déplacement simultané de GPT-2 sur le même device

optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Début de l'entraînement du modèle custom...")
attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print("Custom Attention model trained on SMS dataset. Sample batch loss:", loss.item())

# ---------------------------------------------------------------------
# PART 5: METRICS & EVALUATION
# ---------------------------------------------------------------------
print("\n--- Part 5: Évaluation et calcul des métriques ---")
import evaluate
import numpy as np

accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision") 
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")

print("\n📊 Évaluation du modèle GPT-2...")
gpt2_preds = []
gpt2_labels = []
model.eval()

for ex in val_tok:
    # Ajout d'une dimension de batch .unsqueeze(0)
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    "accuracy":  accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)["accuracy"],
    "precision": precision.compute(predictions=gpt2_preds, references=gpt2_labels)["precision"],
    "recall":    recall.compute(predictions=gpt2_preds, references=gpt2_labels)["recall"],
    "f1":        f1.compute(predictions=gpt2_preds, references=gpt2_labels)["f1"]
}
print("GPT-2 Metrics:", gpt2_metrics)

print("\n📊 Évaluation du modèle Custom Attention...")
attn_preds = []
attn_labels = []
attn_model.eval()

for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    "accuracy":  accuracy.compute(predictions=attn_preds, references=attn_labels)["accuracy"],
    "precision": precision.compute(predictions=attn_preds, references=attn_labels)["precision"],
    "recall":    recall.compute(predictions=attn_preds, references=attn_labels)["recall"],
    "f1":        f1.compute(predictions=attn_preds, references=attn_labels)["f1"]
}
print("Attention Model Metrics:", attn_metrics)